# Text and Audio Preprocessing for English Language

In [1]:
#Import libraries
import os
import re
import unicodedata
import pandas as pd
import numpy as np
import librosa
from tqdm import tqdm


# CONFIGURATION

# Dataset paths
metadata_file = "/kaggle/input/karen-us-female-tts-dataset/metadata.csv"
wav_dir = "/kaggle/input/karen-us-female-tts-dataset/wav"

# Output folders
output_folder = "/kaggle/working/english"
mel_dir = os.path.join(output_folder, "melspectrograms")

os.makedirs(mel_dir, exist_ok=True)

# Metadata outputs
cleaned_metadata_file = os.path.join(output_folder, "eng_metadata_cleaned.tsv")
processed_metadata_file = os.path.join(output_folder, "eng_metadata_processed.tsv")

# Audio config
sample_rate = 22050
n_fft = 1024
hop_length = 256
win_length = 1024
n_mels = 80
fmin = 0
fmax = 8000


# TEXT CLEANING


abbreviations = {
    "mr.": "mister",
    "mrs.": "misses",
    "ms.": "miss",
    "dr.": "doctor",
    "prof.": "professor",
    "e.g": "for example",
    "i.e": "that is",
    "etc.": "and so on",
    "vs.": "versus",
    "st.": "street",
    "ave.": "avenue",
    "rd.": "road",
    "mt.": "mount",
    "lt.": "lieutenant",
    "capt.": "captain",
    "col.": "colonel",
    "gen.": "general",
    "sgt.": "sergeant",
    "sr.": "senior",
    "jr.": "junior",
    "a.m.": "morning",
    "p.m.": "evening",
    "inc.": "incorporated",
    "ltd.": "limited",
    "co.": "company",
    "corp.": "corporation"
}

def expand_abbreviations(text):
    for abbr, full in abbreviations.items():
        text = text.replace(abbr, full)
    return text

def normalize_unicode(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

def clean_text(text):
    text = normalize_unicode(text).strip().lower()
    text = re.sub(r"[^a-zA-Z0-9.,!?'\s-]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = expand_abbreviations(text)
    return text


# LOAD & CLEAN METADATA

df = pd.read_csv(metadata_file, sep="|", names=["audio_id", "text"])
df = df.dropna(subset=["audio_id", "text"])
df["audio_id"] = df["audio_id"].astype(str).str.strip()
df["text"] = df["text"].astype(str).str.strip()
df["cleaned_text"] = df["text"].apply(clean_text)

# Remove empty or duplicate rows
df = df[df["text"].str.strip().astype(bool)]
df = df.drop_duplicates(subset=["audio_id", "text"])

# Save cleaned metadata
df[["audio_id", "cleaned_text"]].to_csv(cleaned_metadata_file, sep="|", index=False, header=False)
print("Cleaned metadata saved at:", cleaned_metadata_file)


# SPECTROGRAM FUNCTIONS

def wav_to_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax
    )
    return librosa.power_to_db(mel, ref=np.max)

# AUDIO PREPROCESSING

metadata_output = []

with open(cleaned_metadata_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Processing audio files...")

for line in tqdm(lines):
    wav_id, text = line.strip().split("|")
    
    # Standardize filename: match dataset exactly
    wav_filename = wav_id
    if not wav_filename.endswith(".wav"):
        wav_filename += ".wav"
    
    wav_path = os.path.join(wav_dir, wav_filename)
    
    # Skip missing files
    if not os.path.exists(wav_path):
        print("Missing audio:", wav_filename)
        continue
    
    # Load audio
    y, sr = librosa.load(wav_path, sr=sample_rate)
    
    # Trim silence
    y, _ = librosa.effects.trim(y, top_db=20)
    
    # Normalize
    y = y / (np.max(np.abs(y)) + 1e-9)
    
    # Generate spectrograms
    mel = wav_to_mel(y)
    
    # Save numpy arrays
    mel_path = os.path.join(mel_dir, wav_id + ".npy")
    
    np.save(mel_path, mel)
    
    # Append to processed metadata
    mel_rel = os.path.join("melspectrograms", wav_id + ".npy")
    
    metadata_output.append(f"{wav_id}|{mel_rel}|{text}")

# Save processed metadata
with open(processed_metadata_file, "w", encoding="utf-8") as f:
    f.write("\n".join(metadata_output))

print("Audio processing complete!")
print("Processed metadata saved at:", processed_metadata_file)


Cleaned metadata saved at: /kaggle/working/english/eng_metadata_cleaned.tsv
Processing audio files...


  0%|          | 0/6847 [00:00<?, ?it/s]

Missing audio: 1.wav
Missing audio: 2.wav
Missing audio: 3.wav
Missing audio: 6.wav
Missing audio: 7.wav
Missing audio: 8.wav
Missing audio: 9.wav
Missing audio: 11.wav
Missing audio: 12.wav
Missing audio: 13.wav
Missing audio: 14.wav
Missing audio: 15.wav
Missing audio: 16.wav
Missing audio: 18.wav
Missing audio: 19.wav
Missing audio: 20.wav
Missing audio: 21.wav
Missing audio: 22.wav
Missing audio: 23.wav
Missing audio: 24.wav
Missing audio: 26.wav
Missing audio: 27.wav
Missing audio: 29.wav
Missing audio: 30.wav
Missing audio: 31.wav
Missing audio: 32.wav
Missing audio: 33.wav
Missing audio: 35.wav
Missing audio: 36.wav
Missing audio: 37.wav
Missing audio: 38.wav
Missing audio: 40.wav
Missing audio: 41.wav
Missing audio: 43.wav
Missing audio: 44.wav
Missing audio: 47.wav
Missing audio: 48.wav
Missing audio: 49.wav
Missing audio: 50.wav
Missing audio: 52.wav
Missing audio: 53.wav
Missing audio: 54.wav
Missing audio: 55.wav
Missing audio: 56.wav
Missing audio: 57.wav
Missing audio: 58

100%|██████████| 6847/6847 [02:53<00:00, 39.48it/s]

Audio processing complete!
Processed metadata saved at: /kaggle/working/english/eng_metadata_processed.tsv


# Text and Audio Preprocessing for Sinhala Language

In [2]:
import re
import pandas as pd
import os
import librosa
import numpy as np
from tqdm import tqdm


# SINHALA CHARACTER SET

vowels = ['අ','ආ','ඇ','ඈ','ඉ','ඊ','උ','ඌ','ඍ','ඎ','එ','ඒ','ඓ','ඔ','ඕ','ඖ']
vowel_strokes = ['්','ා','ැ','ෑ','ි','ී','ු','ූ','ෘ','ෲ','ෙ','ේ','ෛ','ො','ෝ','ෞ','ෟ','ෳ']
consonants = ['ක','ඛ','ග','ඝ','ඞ','ඟ','ච','ඡ','ජ','ඣ','ඤ','ඥ','ඦ','ට','ඨ','ඩ','ඪ','ණ',
              'ත','ථ','ද','ධ','න','ප','ඵ','බ','භ','ම','ය','ර','ල','ව','ශ','ෂ','ස','හ','ළ','ෆ','ට','ඨ']
sinhala_chars = set(vowels + vowel_strokes + consonants)


# PATHS

metadata_file = "/kaggle/input/multi-speaket-tts-dataset-sinhala/file_index.tsv" 
wav_dir = "/kaggle/input/multi-speaket-tts-dataset-sinhala"

# Output folders
output_folder = "/kaggle/working/sinhala"
mel_dir = os.path.join(output_folder, "melspectrograms")
os.makedirs(mel_dir, exist_ok=True)

cleaned_metadata_file = os.path.join(output_folder, "sin_metadata_cleaned.tsv")
processed_metadata_file = os.path.join(output_folder, "sin_metadata_processed.tsv")

# LOAD METADATA

df = pd.read_csv(metadata_file, sep="\t", names=["sentence","file_path"], encoding="utf-8", header=0)
df["file_path"] = df['file_path'].fillna('')
df['file_path'] = df['file_path'].apply(lambda x: os.path.join(wav_dir, os.path.basename(x.strip())))

# Remove empty / duplicate rows
df = df.dropna(subset=["sentence","file_path"])
df = df[df["sentence"].str.strip().astype(bool)]
df = df.drop_duplicates(subset=["sentence","file_path"])


# TEXT NORMALIZATION

def normalize_currency(text):
    text = re.sub(r"\bRs\.?\b", "රුපියල්", text)  
    text = re.sub(r"\$", "ඩොලර්", text)
    text = re.sub(r"£", "පවුම්", text)
    return text

abbreviations = {
    "ප.ව." : "පස්වරු",
    "පෙ.ව." : "පෙරවරු"
}

def normalize_abbreviations(text):
    for abbr, pron in abbreviations.items():
        text = re.sub(re.escape(abbr), pron, text)
    return text

def clean_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = normalize_currency(text)
    text = normalize_abbreviations(text)
    return text

df['cleaned_text'] = df['sentence'].apply(clean_text)

# Save cleaned metadata
df_cleaned = pd.DataFrame({
    "audio_path": df['file_path'].apply(lambda x: os.path.basename(x.strip())),
    "text": df['cleaned_text']
})
df_cleaned.to_csv(cleaned_metadata_file, sep="|", index=False, header=False)
print("Cleaned metadata saved:", cleaned_metadata_file)


# AUDIO PREPROCESSING

sample_rate = 22050
n_fft = 1024
hop_length = 256
win_length = 1024
n_mels = 80
fmin = 0
fmax = 8000

def wav_to_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y, sr=sample_rate, n_fft=n_fft, hop_length=hop_length,
        win_length=win_length, n_mels=n_mels, fmin=fmin, fmax=fmax
    )
    return librosa.power_to_db(mel, ref=np.max)


# Process audio
metadata_output = []

with open(cleaned_metadata_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Processing Sinhala audio...")

for line in tqdm(lines):
    wav_id, text = line.strip().split("|")
    base_id = os.path.splitext(wav_id)[0]

    wav_path = os.path.join(wav_dir, wav_id)
    if not wav_path.endswith(".wav"):
        wav_path += ".wav"
    
    if not os.path.exists(wav_path):
        print("Missing audio:", wav_id)
        continue
    
    y, sr = librosa.load(wav_path, sr=sample_rate)
    y, _ = librosa.effects.trim(y, top_db=20)
    y = y / (np.max(np.abs(y)) + 1e-9)

    mel = wav_to_mel(y)

    mel_path = os.path.join(mel_dir, base_id + ".npy")
    np.save(mel_path, mel)

    mel_rel = os.path.join("melspectrograms", base_id + ".npy")
    metadata_output.append(f"{base_id}|{mel_rel}|{text}")

# Save processed metadata
with open(processed_metadata_file, "w", encoding="utf-8") as f:
    f.write("\n".join(metadata_output))

print("Sinhala audio preprocessing complete!")


Cleaned metadata saved: /kaggle/working/sinhala/sin_metadata_cleaned.tsv
Processing Sinhala audio...


100%|██████████| 5364/5364 [02:17<00:00, 39.00it/s]

Sinhala audio preprocessing complete!


# Text and Audio Preprocessing for Tamil Language

In [3]:
import re
import pandas as pd
import os
import librosa
import numpy as np
from tqdm import tqdm
import unicodedata



# TAMIL CHARACTER SET

vowels = ['அ','ஆ','இ','ஈ','உ','ஊ','எ','ஏ','ஐ','ஒ','ஓ','ஔ']
vowel_strokes = ['ா','ி','ீ','ு','ூ','ெ','ே','ை','ொ','ோ','ௌ','்']
consonants = [
    'க','ங','ச','ஞ','ட','ண','த','ந','ப','ம','ய','ர','ல','வ',
    'ழ','ள','ற','ன','ஷ','ஜ','ஹ','க்ஷ'
]

tamil_chars = set(vowels + vowel_strokes + consonants)



# PATHS

metadata_file = "/kaggle/input/tamil-dataset/line_index.tsv"
wav_dir = "/kaggle/input/tamil-dataset"

# Output folders
output_folder = "/kaggle/working/tamil"
mel_dir = os.path.join(output_folder, "melspectrograms")
os.makedirs(mel_dir, exist_ok=True)

cleaned_metadata_file = os.path.join(output_folder, "tam_metadata_cleaned.tsv")
processed_metadata_file = os.path.join(output_folder, "tam_metadata_processed.tsv")



# LOAD METADATA

df = pd.read_csv(
    metadata_file,
    sep="\t",
    names=["file_path", "sentence"],
    encoding="utf-8"
)

df["file_path"] = df["file_path"].fillna("")
df["file_path"] = df["file_path"].apply(
    lambda x: os.path.join(wav_dir, os.path.basename(x.strip()))
)

# Remove empty / duplicate rows
df = df.dropna(subset=["sentence", "file_path"])
df = df[df["sentence"].str.strip().astype(bool)]
df = df.drop_duplicates(subset=["sentence", "file_path"])



# TEXT NORMALIZATION

def normalize_currency(text):
    text = re.sub(r"\bRs\.?\b", "ரூ", text)
    text = re.sub(r"\$", "டாலர்", text)
    text = re.sub(r"£", "பவுண்டு", text)
    return text

def clean_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = normalize_currency(text)
    return text

df["cleaned_text"] = df["sentence"].apply(clean_text)



# SAVE CLEANED METADATA

df_cleaned = pd.DataFrame({
    "audio_path": df["file_path"].apply(lambda x: os.path.basename(x.strip())),
    "text": df["cleaned_text"]
})

df_cleaned.to_csv(
    cleaned_metadata_file,
    sep="|",
    index=False,
    header=False,
    encoding="utf-8"
)

print("Cleaned Tamil metadata saved:", cleaned_metadata_file)


# AUDIO PREPROCESSING CONFIG


sample_rate = 22050
n_fft = 1024
hop_length = 256
win_length = 1024
n_mels = 80
fmin = 0
fmax = 8000

def wav_to_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax
    )
    return librosa.power_to_db(mel, ref=np.max)


# PROCESS AUDIO FILES

metadata_output = []

with open(cleaned_metadata_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Processing Tamil audio...")

for line in tqdm(lines):
    wav_id, text = line.strip().split("|")
    base_id = os.path.splitext(wav_id)[0]

    wav_path = os.path.join(wav_dir, wav_id)
    wav_path = unicodedata.normalize("NFC", wav_path)

    if not wav_path.endswith(".wav"):
        wav_path += ".wav"

    if not os.path.exists(wav_path):
        print("Missing audio:", wav_id)
        continue

    # Load & normalize audio
    y, sr = librosa.load(wav_path, sr=sample_rate)
    y, _ = librosa.effects.trim(y, top_db=20)
    y = y / (np.max(np.abs(y)) + 1e-9)

    # Mel spectrogram
    mel = wav_to_mel(y)

    # Save mel
    mel_path = os.path.join(mel_dir, base_id + ".npy")
    np.save(mel_path, mel)

    mel_rel = os.path.join("melspectrograms", base_id + ".npy")
    metadata_output.append(f"{base_id}|{mel_rel}|{text}")



# SAVE PROCESSED METADATA

with open(processed_metadata_file, "w", encoding="utf-8") as f:
    f.write("\n".join(metadata_output))

print("Tamil audio preprocessing complete!")


Cleaned Tamil metadata saved: /kaggle/working/tamil/tam_metadata_cleaned.tsv
Processing Tamil audio...


100%|██████████| 2328/2328 [01:08<00:00, 34.01it/s]

Tamil audio preprocessing complete!
